# DOW-30 Walk-Forward Backtest

Extension of `pilot_dow_3_Backtest.ipynb` — works with windowed models from `dow_2_train.ipynb`.

- Reads models trained per window × dataset × algo × seed
- VecNormalize statistics loaded from `{run_name}_vecnorm.pkl`
- Aggregates 5 seeds → mean ± std equity curves
- Baselines: DJIA + MVO (pilot approach: raw % returns, 50% weight cap)

In [1]:
%matplotlib inline
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stable_baselines3 import A2C, PPO, SAC

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent

sys.path.insert(0, str(Path(os.getcwd())))
from utils.config import SEEDS, WINDOWS
from utils.common import (
    make_window_data, make_env_kwargs, get_indicators,
    run_prediction_vecnorm, process_df_for_mvo, stock_returns_computing,
    build_full_price_wide, run_prediction_vecnorm_reconstructed, actions_to_positions,
)
from utils.plots import plot_equity_curves_with_std

In [2]:
FINRL_PATH     = Path(os.environ["FINRL_PATH"])
MODELS_DIR     = FINRL_PATH / "models" / "dow" / "experiment_1"
CACHE_NAME   = "backtest_cache_mv2boll.pkl"
""" 
DATASETS = [
    "all", 'mvxall', 'bollxall', 'mvx2all', "mv2",
    "mv", "mvx", "mvx2", "boll", "bollx", "dbmvx", "3mvx1", '3mvx2',  'bollrsix1', 'bollrsix2',
    "flt_mvx", "flt_mvx2", "flt_bollx", "bollmvx", "bollmvx2", 'flt_bollmvx2','flt_3mvx1', 
    'flt_3mvx2','flt_bollrsix1', 'flt_bollrsix2', "flt_dbmvx",
    "mvx_trg", "mvx2_trg", "bollx_trg",
    'flt_bollmvx2_trg','flt_3mvx1_trg', 'flt_3mvx2_trg','flt_bollrsix1_trg', 'flt_bollrsix2_trg', 
    "flt_mvx_trg",  "flt_mvx2_trg",  "flt_dbmvx_trg", "flt_bollx_trg"
    ] 
"""


DATASETS = [ "all","mv2", "mvx2","flt_mvx2", "flt_mvx2", "mvx2_trg",  "flt_mvx2_trg", 
    "boll", "bollx", "flt_bollx", "bollx_trg", "flt_bollx_trg" 
]   # 'bollxall', 'mvx2all'

#DATASETS = ["mv2]

ALGOS          = {"a2c": A2C, "ppo": PPO, "sac": SAC}
WINDOWS_TO_RUN = ["w1","w4","w3"]  # extend to ["w1","w2","w3"] when all windows are trained
DEVICE         = "cpu"

print(f"MODELS_DIR : {MODELS_DIR}")
print(f"SEEDS      : {SEEDS}")
print(f"WINDOWS    : {WINDOWS_TO_RUN}")
print(f"DATASETS   : {len(DATASETS)}")

# ── Backtest results cache ────────────────────────────────────────────────────
EXPERIMENT    = MODELS_DIR.name                           # e.g. "experiment_1"
CACHE_DIR     = FINRL_PATH / "results" / "dow" / EXPERIMENT
CACHE_DIR.mkdir(parents=True, exist_ok=True)
BACKTEST_CACHE = CACHE_DIR / CACHE_NAME
 # set True to re-run the loop even if cache exists
FORCE_RERUN = False
print(f"\nCache file : {BACKTEST_CACHE}")

MODELS_DIR : /home/main/Programming/finrl/models/dow/experiment_1
SEEDS      : [42, 123, 456, 789, 1234]
WINDOWS    : ['w1', 'w4', 'w3']
DATASETS   : 12

Cache file : /home/main/Programming/finrl/results/dow/experiment_1/backtest_cache_mv2boll.pkl


## 1 — Load walk-forward window data

In [3]:
data = make_window_data(DATASETS, windows=WINDOWS_TO_RUN, cvtd=False,  splits=["train", "test"])

for w in data:
    for ds in data[w]:
        test_shape = data[w][ds]["test"]["df"].shape
        train_tag  = data[w][ds]["train"]["tag"]
        print(f"  {w}/{ds:12s}  test={test_shape}  tag={train_tag}")

  w1/all           test=(6390, 18)  tag=dow_chlovtd_all_2009-01-01_to_2020-02-01
  w1/mv2           test=(6390, 10)  tag=dow_chlovtd_mv2_2009-01-01_to_2020-02-01
  w1/mvx2          test=(6390, 12)  tag=dow_chlovtd_mvx2_2009-01-01_to_2020-02-01
  w1/flt_mvx2      test=(2460, 12)  tag=dow_chlovtd_flt_mvx2_2009-01-01_to_2020-02-01
  w1/mvx2_trg      test=(6390, 10)  tag=dow_chlovtd_mvx2_trg_2009-01-01_to_2020-02-01
  w1/flt_mvx2_trg  test=(2460, 10)  tag=dow_chlovtd_flt_mvx2_trg_2009-01-01_to_2020-02-01
  w1/boll          test=(6390, 10)  tag=dow_chlovtd_boll_2009-01-01_to_2020-02-01
  w1/bollx         test=(6390, 12)  tag=dow_chlovtd_bollx_2009-01-01_to_2020-02-01
  w1/flt_bollx     test=(3600, 12)  tag=dow_chlovtd_flt_bollx_2009-01-01_to_2020-02-01
  w1/bollx_trg     test=(6390, 10)  tag=dow_chlovtd_bollx_trg_2009-01-01_to_2020-02-01
  w1/flt_bollx_trg  test=(3600, 10)  tag=dow_chlovtd_flt_bollx_trg_2009-01-01_to_2020-02-01
  w4/all           test=(7530, 18)  tag=dow_chlovtd_all_2009-01

## 2 — Run all predictions

`run_prediction_vecnorm` (in `utils/common.py`) mirrors `DRLAgent.DRL_prediction` but wraps the
internal VecEnv with the saved `VecNormalize` statistics before inference.

In [4]:
def prep_df(df):
    """Sort and factorize index as required by StockTradingEnv."""
    df = df.copy().sort_values(["date", "tic"], ignore_index=True)
    df.index = df["date"].factorize()[0]
    return df

In [5]:
import pickle

if not FORCE_RERUN and BACKTEST_CACHE.exists():
    with open(BACKTEST_CACHE, "rb") as f:
        _cache = pickle.load(f)
    results       = _cache["results"]
    actions_store = _cache["actions_store"]
    missing       = _cache.get("missing", [])
    _cache_loaded = True
    print(f"✓ Loaded from cache: {BACKTEST_CACHE}")
    print(f"  Windows : {list(results.keys())}")
    if results:
        print(f"  Datasets: {list(next(iter(results.values())).keys())}")
else:
    if BACKTEST_CACHE.exists():
        print(f"[FORCE_RERUN=True] — ignoring existing cache.")
    else:
        print("No cache found — run the loop below.")
    _cache_loaded = False

✓ Loaded from cache: /home/main/Programming/finrl/results/dow/experiment_1/backtest_cache_mv2boll.pkl
  Windows : ['w1', 'w4', 'w3']
  Datasets: ['all', 'mv2', 'mvx2', 'flt_mvx2', 'mvx2_trg', 'flt_mvx2_trg', 'boll', 'bollx', 'flt_bollx', 'bollx_trg', 'flt_bollx_trg']


In [6]:
# Filtered dataset -> its full-daily sibling, used as the reconstruction calendar/price
# source. "all" is missing 2025-12-31 in the w3 test split (confirmed against the raw
# CSV) -- bollx does not, and bollx/mvx2 have identical date coverage on every
# window/split checked, so bollx alone is a safe shared source for both families.
FLT_CALENDAR_SOURCE = {
    "flt_mvx2": "bollx", "flt_mvx2_trg": "bollx",
    "flt_bollx": "bollx", "flt_bollx_trg": "bollx",
}

#_cache_loaded = False
if not _cache_loaded:
    results      = {}
    missing      = []
    actions_store = {}   # actions_store[window][ds][algo_name] = holdings-levels df (seed=42 representative)

    for window in WINDOWS_TO_RUN:
        test_start, test_end = WINDOWS[window]["test"]
        print(f"\n{'='*64}")
        print(f"Window: {window}  ({test_start} \u2192 {test_end})")
        print(f"{'='*64}")
        results[window]       = {}
        actions_store[window] = {}
        _calendar_cache = {}   # full_ds -> (full_price_wide, full_dates), built once per window

        for ds in DATASETS:
            train_tag  = data[window][ds]["train"]["tag"]
            test_df    = prep_df(data[window][ds]["test"]["df"])
            indicators = get_indicators(ds, test_df)
            env_kwargs = make_env_kwargs(test_df, indicators)

            is_filtered = ds in FLT_CALENDAR_SOURCE
            if is_filtered:
                full_ds = FLT_CALENDAR_SOURCE[ds]
                if full_ds not in _calendar_cache:
                    _calendar_cache[full_ds] = build_full_price_wide(data[window][full_ds]["test"]["df"])
                full_price_wide, full_dates = _calendar_cache[full_ds]

            recon_note = "  (reconstructed -- gap-aware)" if is_filtered else ""
            print(f"\n  [{ds}]  {len(indicators)} indicators{recon_note}")
            results[window][ds]       = {}
            actions_store[window][ds] = {}

            for algo_name, algo_cls in ALGOS.items():
                seed_series = []

                for seed in SEEDS:
                    run_name     = f"{train_tag}_{algo_name}_s{seed}"
                    model_path   = MODELS_DIR / run_name
                    vecnorm_path = MODELS_DIR / f"{run_name}_vecnorm.pkl"

                    if not model_path.with_suffix(".zip").exists():
                        missing.append(str(model_path))
                        continue
                    if not vecnorm_path.exists():
                        print(f"    [WARN] No vecnorm: {vecnorm_path.name}")
                        continue

                    try:
                        if is_filtered:
                            curve, positions_df = run_prediction_vecnorm_reconstructed(
                                model_path, vecnorm_path, algo_cls, test_df, indicators,
                                full_price_wide, full_dates, device=DEVICE,
                            )
                            s = curve.rename(f"s{seed}")
                            if seed == SEEDS[0] and algo_name not in actions_store[window][ds]:
                                actions_store[window][ds][algo_name] = positions_df
                        else:
                            model     = algo_cls.load(str(model_path), device=DEVICE)
                            plain_env = StockTradingEnv(df=test_df, **env_kwargs)
                            df_av, df_actions = run_prediction_vecnorm(model, plain_env, vecnorm_path)
                            s = df_av.set_index("date")["account_value"].rename(f"s{seed}")
                            if seed == SEEDS[0] and algo_name not in actions_store[window][ds]:
                                raw_actions = (
                                    df_actions.set_index("date") if "date" in df_actions.columns
                                    else df_actions
                                )
                                actions_store[window][ds][algo_name] = actions_to_positions(raw_actions)
                        seed_series.append(s)
                        print(f"    {algo_name}_s{seed}  \u2713  final=${s.iloc[-1]:>13,.0f}")
                    except Exception as e:
                        print(f"    [ERROR] {algo_name}_s{seed}: {e}")

                results[window][ds][algo_name] = seed_series

    if missing:
        print(f"\n--- Missing models ({len(missing)}) ---")
        for m in missing[:20]:
            print(f"  {m}")
else:
    print("\n[Cache loaded] \u2014 skipping loop.")


[Cache loaded] — skipping loop.


In [7]:
import pickle
if not _cache_loaded:
    with open(BACKTEST_CACHE, "wb") as f:
        pickle.dump({"results": results, "actions_store": actions_store, "missing": missing}, f)
    print(f"\n✓ Saved to {BACKTEST_CACHE}")
else:
    print("[Cache] Results already in memory — no save needed.")

[Cache] Results already in memory — no save needed.


In [8]:
# --- 5-seed trade-stats sweep ---
# actions_store (cell 8) only ever kept ONE seed's (42) position DataFrame -- the
# other 4 seeds' positions were computed in that same loop and then discarded, never
# cached. There's no way to get 5-seed-averaged trade counts from the existing cache;
# this reruns inference (same trained models, no retraining) to capture positions for
# every seed and compute total_trades per seed, so Trades can be averaged/CI'd the
# same way Return%/Sharpe/MaxDD% already are.
from utils.tables import compute_trade_stats

TRADE_STATS_CACHE = CACHE_DIR / "trade_stats_5seed_cache.pkl"

if not FORCE_RERUN and TRADE_STATS_CACHE.exists():
    with open(TRADE_STATS_CACHE, "rb") as f:
        trade_stats = pickle.load(f)["trade_stats"]
    print(f"✓ Loaded trade-stats cache: {TRADE_STATS_CACHE}")
else:
    trade_stats = {}
    for window in WINDOWS_TO_RUN:
        trade_stats[window] = {}
        _calendar_cache = {}
        for ds in DATASETS:
            train_tag  = data[window][ds]["train"]["tag"]
            test_df    = prep_df(data[window][ds]["test"]["df"])
            indicators = get_indicators(ds, test_df)
            env_kwargs = make_env_kwargs(test_df, indicators)

            is_filtered = ds in FLT_CALENDAR_SOURCE
            if is_filtered:
                full_ds = FLT_CALENDAR_SOURCE[ds]
                if full_ds not in _calendar_cache:
                    _calendar_cache[full_ds] = build_full_price_wide(data[window][full_ds]["test"]["df"])
                full_price_wide, full_dates = _calendar_cache[full_ds]

            trade_stats[window][ds] = {}
            for algo_name, algo_cls in ALGOS.items():
                seed_trades = []
                for seed in SEEDS:
                    run_name     = f"{train_tag}_{algo_name}_s{seed}"
                    model_path   = MODELS_DIR / run_name
                    vecnorm_path = MODELS_DIR / f"{run_name}_vecnorm.pkl"
                    if not model_path.with_suffix(".zip").exists() or not vecnorm_path.exists():
                        continue
                    try:
                        if is_filtered:
                            _, positions_df = run_prediction_vecnorm_reconstructed(
                                model_path, vecnorm_path, algo_cls, test_df, indicators,
                                full_price_wide, full_dates, device=DEVICE,
                            )
                        else:
                            model     = algo_cls.load(str(model_path), device=DEVICE)
                            plain_env = StockTradingEnv(df=test_df, **env_kwargs)
                            _, df_actions = run_prediction_vecnorm(model, plain_env, vecnorm_path)
                            raw_actions = (
                                df_actions.set_index("date") if "date" in df_actions.columns
                                else df_actions
                            )
                            positions_df = actions_to_positions(raw_actions)
                        seed_trades.append(compute_trade_stats(positions_df)["total_trades"])
                    except Exception as e:
                        print(f"    [ERROR] {ds} {algo_name}_s{seed}: {e}")
                trade_stats[window][ds][algo_name] = seed_trades
            print(f"  {window} [{ds:16s}] trades: " +
                  ", ".join(f"{a}={len(trade_stats[window][ds][a])}/{len(SEEDS)}" for a in ALGOS))

    with open(TRADE_STATS_CACHE, "wb") as f:
        pickle.dump({"trade_stats": trade_stats}, f)
    print(f"✓ Saved: {TRADE_STATS_CACHE}")


✓ Loaded trade-stats cache: /home/main/Programming/finrl/results/dow/experiment_1/trade_stats_5seed_cache.pkl


In [9]:
# --- Per-seed metrics table (metrics_seeds_table.csv) ---
# One row per (window, dataset, algo, seed): each seed's curve scored individually via
# compute_metrics (never averaged first), joined with that seed's trade count. This is
# the seed-level counterpart to metrics_table.csv (built later in this notebook, which
# only has the aggregate) -- dow_4_ablations.ipynb reads this directly for every
# *native* condition (a model evaluated on the same kind of sequence it was trained
# on) instead of reaching into this notebook's internal pickles (results/trade_stats).
#
# Curves in `results` carry their real seed in .name ("s42" etc.); trade_stats' lists
# don't carry seed labels, just positional order -- both sweeps skip missing models in
# the identical `for seed in SEEDS: ... continue` pattern, so the two line up seed-for-
# seed whenever nothing was skipped -- checked explicitly below, not assumed.
from utils.tables import compute_metrics

seed_rows = []
for w in results:
    for ds in results[w]:
        for algo in results[w][ds]:
            curves = results[w][ds][algo]
            trades = trade_stats.get(w, {}).get(ds, {}).get(algo, [])
            if trades and len(trades) != len(curves):
                print(f"  [WARN] {w}/{ds}/{algo}: {len(curves)} curves vs {len(trades)} "
                      f"trade counts -- Trades left blank for this cell (seed alignment unclear)")
                trades = []
            for i, curve in enumerate(curves):
                seed = int(curve.name[1:]) if curve.name and curve.name.startswith("s") else SEEDS[i]
                m = compute_metrics(curve)
                seed_rows.append({
                    "window": w, "dataset": ds, "algo": algo, "seed": seed,
                    "Return%": m["Return%"], "Sharpe": m["Sharpe"], "MaxDD%": m["MaxDD%"],
                    "Trades": trades[i] if trades else np.nan,
                })

metrics_seeds_df = pd.DataFrame(seed_rows)
metrics_seeds_path = "tables_mv2boll/metrics_seeds_table.csv"
metrics_seeds_df.to_csv(metrics_seeds_path, index=False)
print(f"✓ Saved: {metrics_seeds_path}  ({len(metrics_seeds_df)} rows)")


✓ Saved: tables_mv2boll/metrics_seeds_table.csv  (495 rows)


## 3 — Aggregate seeds → mean ± std

In [10]:
def aggregate_seeds(results):
    """Returns agg[window][dataset][algo] = {'mean': Series, 'std': Series, 'n': int}"""
    agg = {}
    for window, ds_dict in results.items():
        agg[window] = {}
        for ds, algo_dict in ds_dict.items():
            agg[window][ds] = {}
            for algo, seed_list in algo_dict.items():
                if not seed_list:
                    continue
                mat = pd.concat(seed_list, axis=1)
                agg[window][ds][algo] = {
                    "mean": mat.mean(axis=1),
                    "std":  mat.std(axis=1),
                    "n":    len(seed_list),
                }
    return agg


agg = aggregate_seeds(results)

print("Aggregation summary:")
for w in agg:
    for ds in agg[w]:
        info = {a: agg[w][ds][a]["n"] for a in agg[w][ds]}
        print(f"  {w}/{ds:12s}: {info}")

Aggregation summary:
  w1/all         : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w1/mv2         : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w1/mvx2        : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w1/flt_mvx2    : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w1/mvx2_trg    : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w1/flt_mvx2_trg: {'a2c': 5, 'ppo': 5, 'sac': 5}
  w1/boll        : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w1/bollx       : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w1/flt_bollx   : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w1/bollx_trg   : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w1/flt_bollx_trg: {'a2c': 5, 'ppo': 5, 'sac': 5}
  w4/all         : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w4/mv2         : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w4/mvx2        : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w4/flt_mvx2    : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w4/mvx2_trg    : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w4/flt_mvx2_trg: {'a2c': 5, 'ppo': 5, 'sac': 5}
  w4/boll        : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w4/bollx       : {'a2c': 5, 'ppo': 5, 'sac': 5}
  w4/flt_bollx   : {'a2c': 5

## 4 — Baselines: DJIA, MVO, S&P 500 and Equal-Weight Dow 30 Buy-and-Hold 

MVO uses the same approach as `pilot_dow_3_Backtest.ipynb`:
- Raw period-over-period % returns (not annualised)
- `weight_bounds=(0, 0.5)` (max 50% per stock)
- Weights applied as share quantities computed from last training-period price

In [11]:
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader

def get_djia_baseline(test_start, test_end, initial_amount=1_000_000):
    """Download DJIA and normalise to initial_amount."""
    df = YahooDownloader(
        start_date=test_start, end_date=test_end, ticker_list=["^DJI"]
    ).fetch_data()
    df = df.sort_values("date").set_index("date")
    series = df["close"] / df["close"].iloc[0] * initial_amount
    series.name = "DJIA"
    return series

In [12]:
def compute_mvo_baseline(train_df, test_df, initial_amount=1_000_000):
    """
    MVO baseline matching pilot_dow_3_Backtest.ipynb:
    - process_df_for_mvo / stock_returns_computing from utils/common.py
    - Raw period-over-period % returns (not annualised)
    - EfficientFrontier with weight_bounds=(0, 0.5)
    - Weights expressed as share quantities at the last training price
    """
    from pypfopt.efficient_frontier import EfficientFrontier

    stock_data     = process_df_for_mvo(train_df)
    trade_data_mvo = process_df_for_mvo(test_df)
    stock_dimension = stock_data.shape[1]

    ar_prices          = np.asarray(stock_data)
    rows, cols         = ar_prices.shape
    ar_returns         = stock_returns_computing(ar_prices, rows, cols)
    mean_returns       = np.mean(ar_returns, axis=0)
    cov_returns        = np.cov(ar_returns, rowvar=False)

    ef = EfficientFrontier(mean_returns, cov_returns, weight_bounds=(0, 0.5))
    ef.max_sharpe()
    cleaned_weights    = ef.clean_weights()
    mvo_weights        = np.array([initial_amount * cleaned_weights[i]
                                   for i in range(stock_dimension)])

    last_price         = np.array([1 / p for p in stock_data.tail(1).to_numpy()[0]])
    initial_portfolio  = np.multiply(mvo_weights, last_price)
    portfolio_assets   = trade_data_mvo @ initial_portfolio

    # weight_bounds=(0, 0.5) lets the optimizer zero out tickers entirely; clean_weights()
    # already rounds sub-cutoff weights to exactly 0.0, so this is an exact count of how
    # many tickers actually got bought -- the one discrete "trade" event this static
    # buy-and-hold portfolio ever makes (bought once, held unchanged to the end).
    n_trades = sum(w > 0 for w in cleaned_weights.values())
    return pd.DataFrame(
        portfolio_assets.values,
        index=pd.to_datetime(portfolio_assets.index),
        columns=["MVO"],
    ), n_trades

In [13]:
baselines = {}
# Trades per (window, benchmark name) for the baselines that have a real, computable
# trade count -- a static portfolio bought once and held. DJIA/SP500 (cells below) are
# pure index price series with no simulated portfolio at all, so no entry is ever added
# for them here; metrics_table.csv's merge later leaves those rows' Trades as NaN,
# which is the honest answer (see the analysis discussion for this feature).
baseline_trades = {}

for window in WINDOWS_TO_RUN:
    test_start, test_end = WINDOWS[window]["test"]
    baselines[window] = {}
    baseline_trades[window] = {}

    # DJIA
    try:
        djia = get_djia_baseline(test_start, test_end)
        baselines[window]["DJIA"] = djia
        print(f"{window} DJIA: {djia.iloc[0]:>12,.0f} \u2192 {djia.iloc[-1]:>12,.0f}")
    except Exception as e:
        print(f"[WARN] DJIA failed: {e}")

    # MVO -- use 'bollx' as the price source. "all" is missing 2025-12-31 in the w3
    # test split (confirmed against the raw CSV); bollx does not and is otherwise
    # pixel-identical to "all" on close price, so this only affects w3 accuracy.
    try:
        train_df = data[window]["bollx"]["train"]["df"]
        test_df  = data[window]["bollx"]["test"]["df"]
        mvo, mvo_trades = compute_mvo_baseline(train_df, test_df)
        baselines[window]["MVO"] = mvo
        baseline_trades[window]["MVO"] = mvo_trades
        print(f"{window} MVO : {mvo.iloc[0, 0]:>12,.0f} \u2192 {mvo.iloc[-1, 0]:>12,.0f}  ({mvo_trades} trades)")
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"[WARN] MVO failed: {e}")

[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (213, 8)
w1 DJIA:    1,000,000 →    1,146,168


w1 MVO :      981,669 →    1,368,471  (9 trades)


[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (251, 8)
w4 DJIA:    1,000,000 →      906,032


w4 MVO :    1,045,648 →      751,705  (6 trades)


[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (250, 8)
w3 DJIA:    1,000,000 →    1,133,775


w3 MVO :      970,254 →    1,053,041  (10 trades)


In [14]:
# ── New benchmarks: S&P 500 and Equal-Weight Dow 30 Buy-and-Hold ─────────
from utils.common import get_etf_baseline, compute_ew_bh

summary_rows = []

for window in WINDOWS_TO_RUN:
    test_start, test_end = WINDOWS[window]["test"]

    # ── S&P 500 (^GSPC price return, adjusted close) ──────────────────────
    try:
        sp500 = get_etf_baseline("^GSPC", test_start, test_end)
        baselines[window]["SP500"] = sp500
        ret_sp = (sp500.iloc[-1] / sp500.iloc[0] - 1) * 100
        print(f"{window}  SP500     : {ret_sp:+.2f}%"
              f"  ({sp500.index[0].date()} → {sp500.index[-1].date()})")
        summary_rows.append({"Window": window, "Benchmark": "SP500",
                              "Return%": round(ret_sp, 2)})
    except Exception as e:
        print(f"[WARN] SP500 failed for {window}: {e}")

    # ── Equal-weight Dow 30 buy-and-hold (same universe as RL experiments) ─
    try:
        # "all" is missing 2025-12-31 in the w3 test split (confirmed against the raw CSV);
        # bollx does not, so it is used as the price source here too.
        test_df_ew = data[window]["bollx"]["test"]["df"]
        ew_curve, ret_ew, ew_missing = compute_ew_bh(test_df_ew)
        baselines[window]["EW_DJ30_BH"] = ew_curve
        n_stocks = test_df_ew["tic"].nunique()
        baseline_trades[window]["EW_DJ30_BH"] = n_stocks
        miss_str = f", missing: {ew_missing}" if ew_missing else ""
        print(f"{window}  EW_DJ30_BH: {ret_ew:+.2f}%"
              f"  ({n_stocks} stocks{miss_str})")
        summary_rows.append({"Window": window, "Benchmark": "EW_DJ30_BH",
                              "Return%": round(ret_ew, 2)})
    except Exception as e:
        print(f"[WARN] EW_DJ30_BH failed for {window}: {e}")

# ── Summary table ─────────────────────────────────────────────────────────
import pandas as pd
summary_df = pd.DataFrame(summary_rows)
print("\n" + summary_df.to_string(index=False))

import os
os.makedirs("tables/all", exist_ok=True)
summary_df.to_csv("tables/all/new_benchmarks.csv", index=False)
print("✓ Saved: tables/all/new_benchmarks.csv")

# ── Verification against reference closing prices ─────────────────────────
REF_SP500 = {"w1": 21.55, "w2": 28.78, "w3": 16.65}
print("\n── S&P 500 verification (ref from GSPC close prices) ──")
for row in summary_rows:
    if row["Benchmark"] == "SP500":
        w = row["Window"]
        ref = REF_SP500.get(w)
        if ref is not None:
            diff = abs(row["Return%"] - ref)
            status = "✓" if diff < 1.0 else "⚠ MISMATCH"
            print(f"  {w}: computed {row['Return%']:+.2f}%  "
                  f"ref {ref:+.2f}%  Δ={diff:.2f}pp  {status}")


[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (213, 8)
w1  SP500     : +21.55%  (2020-03-02 → 2020-12-31)
w1  EW_DJ30_BH: +22.02%  (30 stocks)


[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (251, 8)
w4  SP500     : -19.95%  (2022-01-03 → 2022-12-30)
w4  EW_DJ30_BH: -9.07%  (30 stocks)


[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (250, 8)
w3  SP500     : +16.65%  (2025-01-02 → 2025-12-31)
w3  EW_DJ30_BH: +18.54%  (30 stocks)

Window  Benchmark  Return%
    w1      SP500    21.55
    w1 EW_DJ30_BH    22.02
    w4      SP500   -19.95
    w4 EW_DJ30_BH    -9.07
    w3      SP500    16.65
    w3 EW_DJ30_BH    18.54
✓ Saved: tables/all/new_benchmarks.csv

── S&P 500 verification (ref from GSPC close prices) ──
  w1: computed +21.55%  ref +21.55%  Δ=0.00pp  ✓
  w3: computed +16.65%  ref +16.65%  Δ=0.00pp  ✓


In [15]:
# Equal-weight Dow 30, PERIODICALLY REBALANCED back to equal weight -- unlike the
# static buy-and-hold above (bought once, held unchanged), this one actually trades
# on a fixed calendar schedule, so its Trades count is genuinely comparable in
# magnitude to the RL agents' (the static versions only ever place ~1 trade per
# ticker, total, for the whole test window). Both a monthly and a quarterly variant
# are run -- quarterly matches the real-world S&P Equal Weight Index's own
# rebalancing convention; monthly rebalances more often. cost_bps=0.001 matches the
# RL environment's own buy_cost_pct/sell_cost_pct exactly (see make_env_kwargs in
# utils/common.py) -- pass cost_bps=0 to compare without transaction costs.
from utils.common import compute_bh_dj30

for window in WINDOWS_TO_RUN:
    # Same price source as the SP500/static-EW_BH cell above, for the same reason
    # ("all" is missing 2025-12-31 in the w3 test split; bollx is not).
    test_df_ew = data[window]["bollx"]["test"]["df"]
    for freq in ["M", "Q"]:
        name = f"EW_DJ30_{freq}"
        curve, n_trades = compute_bh_dj30(test_df_ew, freq=freq, cost_bps=0.001)
        baselines[window][name] = curve
        baseline_trades[window][name] = n_trades
        ret_pct = (curve.iloc[-1] / curve.iloc[0] - 1) * 100
        print(f"{window}  {name}: {ret_pct:+.2f}%  ({n_trades} trades)")


w1  EW_DJ30_M: +23.31%  (264 trades)
w1  EW_DJ30_Q: +23.50%  (89 trades)
w4  EW_DJ30_M: -10.69%  (320 trades)
w4  EW_DJ30_Q: -10.15%  (90 trades)


w3  EW_DJ30_M: +20.05%  (318 trades)
w3  EW_DJ30_Q: +19.22%  (89 trades)


## 5 — Plots

**View A**: one figure per algorithm — all datasets as lines (compare indicator sets)  
**View B**: one figure per dataset — all algorithms as lines (compare algorithms)

In [18]:
import os, importlib
import utils.tables as tbl
import utils.plots as plots_module

importlib.reload(tbl)           # pick up any edits to tables.py
importlib.reload(plots_module)  # pick up any edits to plots.py

from utils.plots import (
    plot_trade_heatmap,
    plot_ticker_buysell_aggregated,
    plot_portfolio_comparison,
    plot_top_configs_equity,
    plot_rl_trades_on_price,
    plot_equity_curves_with_std,
)

os.makedirs("tables", exist_ok=True)

SAVE_DIR = "tables/"
SAVE_TABLE = True
SAVE_IMG=False
WINDOWS_TO_RUN = ["w1","w4","w3",]  # extend to ["w1","w2","w3"] when all windows are trained


In [18]:
agg["w2"]["flt_mvx2"]["sac"]["mean"].keys

<bound method Series.keys of date
2021-01-07    1.000000e+06
2021-01-11    9.998298e+05
2021-01-19    9.989755e+05
2021-01-26    1.001640e+06
2021-01-27    9.875683e+05
                  ...     
2021-12-09    1.201532e+06
2021-12-10    1.210362e+06
2021-12-15    1.216347e+06
2021-12-16    1.215745e+06
2021-12-17    1.200808e+06
Length: 106, dtype: float64>

In [ ]:
# View A: per-algo (how do indicator sets compare within one algorithm?)
%matplotlib inline
#WINDOWS_TO_RUN = WINDOWS_TO_RUN[0]


DATASETS = ["all",  'mvx2all', "mv2", "flt_mvx2", "flt_mvx2", "mvx2_trg",  "flt_mvx2_trg" ]

#DATASETS = ["all","'bollxall', boll", "bollx", "flt_bollx", "bollx_trg", "flt_bollx_trg"  ]

for window in WINDOWS_TO_RUN:
    test_start, test_end = WINDOWS[window]["test"]
    bl = baselines.get(window, {})
    for algo in ALGOS:
        plot_equity_curves_with_std(
            agg[window], DATASETS, [algo],
            title=f"{algo.upper()} \u2014 {window.upper()} ({test_start} to {test_end})",
            baselines=bl, save=SAVE_IMG, save_dir=f"imgs/dow/{str(window)}",
        )

In [ ]:
# View A: per-algo (how do indicator sets compare within one algorithm?)
%matplotlib inline
# filter and triggers for moving av. 
DATASETS = ["all","'bollxall', boll", "bollx", "flt_bollx", "bollx_trg", "flt_bollx_trg"  ]

for window in WINDOWS_TO_RUN:
    test_start, test_end = WINDOWS[window]["test"]
    bl = baselines.get(window, {})
    for algo in ALGOS:
        plot_equity_curves_with_std(
            agg[window], DATASETS, [algo],
            title=f"{algo.upper()} \u2014 {window.upper()} ({test_start} to {test_end})",
            baselines=bl, save=False,
        )

## 6 — Analysis Tables

Metrics, interaction heatmaps, filter-effect comparison, hypothesis verdicts, and rankings.
All tables can be saved to `tables/` as CSV.

In [16]:
import os, importlib
import utils.tables as tbl
import utils.plots as plots_module

importlib.reload(tbl)           # pick up any edits to tables.py
importlib.reload(plots_module)  # pick up any edits to plots.py

from utils.plots import (
    plot_trade_heatmap,
    plot_ticker_buysell_aggregated,
    plot_portfolio_comparison,
    plot_top_configs_equity,
    plot_rl_trades_on_price,
)

os.makedirs("tables", exist_ok=True)

SAVE_DIR = "tables_mv2boll"
SAVE_TABLE = True


In [41]:
DATASETS = [
    "all", 'mvxall', 'bollxall', 'mvx2all',
    "mv", "mvx", "mvx2", "boll", "bollx", "dbmvx", "3mvx1", '3mvx2',  'bollrsix1', 'bollrsix2',
    "flt_mvx", "flt_mvx2", "flt_bollx", "bollmvx", "bollmvx2", 'flt_bollmvx2','flt_3mvx1', 
    'flt_3mvx2','flt_bollrsix1', 'flt_bollrsix2', "flt_dbmvx",
    "mvx_trg", "mvx2_trg", "bollx_trg",
    'flt_bollmvx2_trg','flt_3mvx1_trg', 'flt_3mvx2_trg','flt_bollrsix1_trg', 'flt_bollrsix2_trg', 
    "flt_mvx_trg",  "flt_mvx2_trg",  "flt_dbmvx_trg", "flt_bollx_trg"
    ]

In [18]:
agg["w1"]["mv2"]

{'a2c': {'mean': date
  2020-03-02    1.000000e+06
  2020-03-03    9.950869e+05
  2020-03-04    1.009435e+06
  2020-03-05    9.941420e+05
  2020-03-06    9.857245e+05
                    ...     
  2020-12-24    1.259324e+06
  2020-12-28    1.270533e+06
  2020-12-29    1.267715e+06
  2020-12-30    1.270072e+06
  2020-12-31    1.276653e+06
  Length: 213, dtype: float64,
  'std': date
  2020-03-02        0.000000
  2020-03-03      986.631157
  2020-03-04     2680.452358
  2020-03-05     2594.551190
  2020-03-06     2974.169086
                    ...     
  2020-12-24    57836.912701
  2020-12-28    61378.843658
  2020-12-29    61505.307728
  2020-12-30    64655.546649
  2020-12-31    67484.883278
  Length: 213, dtype: float64,
  'n': 5},
 'ppo': {'mean': date
  2020-03-02    1.000000e+06
  2020-03-03    9.994692e+05
  2020-03-04    1.000889e+06
  2020-03-05    9.990827e+05
  2020-03-06    9.984110e+05
                    ...     
  2020-12-24    1.290473e+06
  2020-12-28    1.301433e+06

In [17]:
DATASETS = ["all", "boll", "bollx", "flt_bollx", "bollx_trg", "flt_bollx_trg",  
                    "mv2", "mvx2", "flt_mvx2", "mvx2_trg",  "flt_mvx2_trg" ]

In [18]:
# --- 6.1  Full metrics table (all windows × datasets × algos + DJIA / MVO) ---
metrics_df = tbl.make_metrics_table(
    agg, WINDOWS_TO_RUN, DATASETS, list(ALGOS),
    baselines=baselines, save=SAVE_TABLE, save_dir=SAVE_DIR,
    results=results,
)

# Merge in 5-seed-averaged trade counts (trade_stats, from the sweep cell above) --
# a post-processing merge here in dow_3 only, no change to make_metrics_table itself
# (shared with dow_4/dow_5). Reuses tables._t_crit so the CI math matches
# _seed_ci_stats' formula exactly, just applied to trade counts (scalars) instead of
# equity curves.
from utils.tables import _t_crit

def _trades_ci(seed_trades):
    if not seed_trades:
        return dict(Trades_mean=np.nan, Trades_std=np.nan, Trades_ci_lo=np.nan, Trades_ci_hi=np.nan)
    vals = np.array(seed_trades, dtype=float)
    n = len(vals)
    mean = vals.mean()
    if n < 2:
        return dict(Trades_mean=round(float(mean), 3), Trades_std=np.nan,
                     Trades_ci_lo=np.nan, Trades_ci_hi=np.nan)
    std = vals.std(ddof=1)
    se = std / np.sqrt(n)
    t = _t_crit(n)
    return dict(Trades_mean=round(float(mean), 3), Trades_std=round(float(std), 3),
                Trades_ci_lo=round(float(mean - t * se), 3), Trades_ci_hi=round(float(mean + t * se), 3))

trades_rows = []
for w in trade_stats:
    for ds in trade_stats[w]:
        for algo in trade_stats[w][ds]:
            row = {"window": w, "dataset": ds, "algo": algo}
            row.update(_trades_ci(trade_stats[w][ds][algo]))
            trades_rows.append(row)
trades_df = pd.DataFrame(trades_rows).set_index(["window", "dataset", "algo"])

metrics_df = metrics_df.merge(trades_df, left_index=True, right_index=True, how="left")

# Fill in Trades for the baselines that have one (MVO/EW_DJ30_BH, from the baselines
# cells above) -- aligns by the same (window, "[name]", "—") index make_metrics_table
# already uses for baseline rows, so only those rows' still-NaN Trades_mean gets filled.
# DJIA/SP500 have no entry in baseline_trades (no simulated portfolio), so they stay NaN.
baseline_trades_series = pd.Series({
    (w, f"[{bname}]", "—"): n
    for w, bmap in baseline_trades.items() for bname, n in bmap.items()
}).rename_axis(["window", "dataset", "algo"])
metrics_df["Trades_mean"] = metrics_df["Trades_mean"].fillna(baseline_trades_series)

if SAVE_TABLE:
    metrics_df.to_csv(f"{SAVE_DIR}/metrics_table.csv")
    print(f"✓ Saved (with Trades columns): {SAVE_DIR}/metrics_table.csv")
metrics_df


Saved: tables_mv2boll/metrics_table.csv
✓ Saved (with Trades columns): tables_mv2boll/metrics_table.csv


Return%   CAGR  Sharpe  Sortino  Calmar  MaxDD%  \
window dataset      algo                                                    
w1     all          a2c     26.23  31.73   0.935    1.312   1.216  -26.24   
                    ppo     33.21  40.39   1.755    2.517   4.424   -9.18   
                    sac     18.47  22.21   0.729    1.023   0.823  -27.11   
       boll         a2c     24.26  29.30   0.871    1.207   1.068  -27.58   
                    ppo     35.19  42.86   1.785    2.579   4.223  -10.21   
...                           ...    ...     ...      ...     ...     ...   
w3     [MVO]        —        8.53   8.60   0.502    0.745   0.445  -19.40   
       [SP500]      —       16.65  16.79   0.922    1.371   0.892  -18.90   
       [EW_DJ30_BH] —       18.54  18.70   1.136    1.707   1.175  -15.99   
       [EW_DJ30_M]  —       20.05  20.22   1.183    1.794   1.237  -16.42   
       [EW_DJ30_Q]  —       19.22  19.39   1.151    1.743   1.202  -16.20   

                          Volatility%  Corr_DJIA  Corr_MVO  Corr_SP500  ...  \
window dataset      algo                                                ...   
w1     all          a2c         36.94      0.983     0.918       0.974  ...   
                    ppo         20.64      0.802     0.755       0.803  ...   
                    sac         37.13      0.980     0.882       0.956  ...   
       boll         a2c         37.95      0.984     0.927       0.980  ...   
                    ppo         21.36      0.746     0.678       0.734  ...   
...                               ...        ...       ...         ...  ...   
w3     [MVO]        —           20.70        NaN     1.000         NaN  ...   
       [SP500]      —           18.79        NaN       NaN         NaN  ...   
       [EW_DJ30_BH] —           16.31        NaN       NaN         NaN  ...   
       [EW_DJ30_M]  —           16.82        NaN       NaN         NaN  ...   
       [EW_DJ30_Q]  —           16.64        NaN       NaN         NaN  ...   

                          Sharpe_ci_hi  MaxDD%_n  MaxDD%_mean  MaxDD%_std  \
window dataset      algo                                                    
w1     all          a2c          1.233         5      -26.238       4.601   
                    ppo          2.001         5       -9.938       2.215   
                    sac          0.914         5      -27.110       2.840   
       boll         a2c          1.039         5      -27.584       3.087   
                    ppo          1.924         5      -11.170       1.159   
...                                ...       ...          ...         ...   
w3     [MVO]        —              NaN         0          NaN         NaN   
       [SP500]      —              NaN         0          NaN         NaN   
       [EW_DJ30_BH] —              NaN         0          NaN         NaN   
       [EW_DJ30_M]  —              NaN         0          NaN         NaN   
       [EW_DJ30_Q]  —              NaN         0          NaN         NaN   

                          MaxDD%_ci_lo  MaxDD%_ci_hi  Trades_mean  Trades_std  \
window dataset      algo                                                        
w1     all          a2c        -31.951       -20.525        144.4      38.920   
                    ppo        -12.688        -7.188       2314.8     621.130   
                    sac        -30.636       -23.584        794.2     170.036   
       boll         a2c        -31.417       -23.751        107.6      42.647   
                    ppo        -12.610        -9.730       1283.4     348.244   
...                                ...           ...          ...         ...   
w3     [MVO]        —              NaN           NaN         10.0         NaN   
       [SP500]      —              NaN           NaN          NaN         NaN   
       [EW_DJ30_BH] —              NaN           NaN         30.0         NaN   
       [EW_DJ30_M]  —              NaN           NaN        318.0         NaN   
       [EW_DJ30_Q]  — 

In [47]:
# --- 6.2  Interaction heatmap — Sharpe (all datasets × algos, per window) ---
for w in WINDOWS_TO_RUN:
    print(f"\n=== {w.upper()} — Sharpe Heatmap (all datasets) ===")
    hm = tbl.make_interaction_heatmap(
        agg, w, DATASETS, list(ALGOS),
        metric="sharpe", metrics_df=metrics_df, save=SAVE_TABLE , save_dir = SAVE_DIR +f"/{w}",
    )
    display(hm.style
              .background_gradient(cmap="RdYlGn", axis=None)
              .format("{:.3f}", na_rep="—"))


=== W1 — Sharpe Heatmap (all datasets) ===


/home/main/Dropbox/phd/ResearchFinRL/FinRLCode/myresearch/tables.py:368: PerformanceWarning: indexing past lexsort depth may impact performance.
  val = metrics_df.loc[(window, ds, algo), col_name]


ValueError: Incompatible indexer with Series

In [ ]:
# --- 6.4  Filter effect table (flt_* vs base, Δ Return%/Sharpe/MaxDD%/Trades, paired 95% CI) ---
FILTER_PAIRS = [(b, f) for b, f in [
    ("mvx2",  "flt_mvx2"),
    ("bollx", "flt_bollx"),
   ("mvx2_trg",  "flt_mvx2_trg"),
    ("bollx_trg", "flt_bollx_trg"),
] if b in DATASETS and f in DATASETS]

from scipy import stats as _stats

def _paired_ci(diffs):
    diffs = np.array(diffs, dtype=float)
    n = len(diffs)
    mean = diffs.mean()
    se = diffs.std(ddof=1) / np.sqrt(n)
    t = _stats.t.ppf(0.975, n - 1)
    return mean, mean - t * se, mean + t * se, int((diffs > 0).sum()), n

# Reads the *_mean columns (per-seed metric, then averaged) -- not the plain
# Return%/Sharpe/MaxDD% columns, which come from averaging the 5 seed curves together
# before computing the metric once (optimistic Sharpe, shallow MaxDD -- see the
# aggregation-bias fix applied elsewhere in this project). Trades has no "plain"
# variant at all -- Trades_mean is the only Trades column.
METRICS_FOR_FILTER = {"Return%": "Return%_mean", "Sharpe": "Sharpe_mean",
                       "MaxDD%": "MaxDD%_mean", "Trades": "Trades_mean"}

# One row per pair, pooled to n=9 (3 windows x 3 algos) -- same paired-CI granularity
# used throughout this project's other significance tests. Saved once, no per-window
# folders.
filter_rows = []
for base_ds, flt_ds in FILTER_PAIRS:
    diffs_by_metric = {m: [] for m in METRICS_FOR_FILTER}
    for w in WINDOWS_TO_RUN:
        for algo in ALGOS:
            try:
                base_row = metrics_df.loc[(w, base_ds, algo)]
                flt_row  = metrics_df.loc[(w, flt_ds, algo)]
            except KeyError:
                continue
            for m, col in METRICS_FOR_FILTER.items():
                b, f = base_row.get(col), flt_row.get(col)
                if pd.notna(b) and pd.notna(f):
                    diffs_by_metric[m].append(f - b)

    row = {"base": base_ds, "filtered": flt_ds}
    for m, diffs in diffs_by_metric.items():
        if len(diffs) < 2:
            row[f"Δ_{m}"] = np.nan
            row[f"Δ_{m}_ci_lo"] = np.nan
            row[f"Δ_{m}_ci_hi"] = np.nan
            row[f"Δ_{m}_reliable"] = False
            row[f"Δ_{m}_n"] = len(diffs)
            continue
        mean, lo, hi, wins, n = _paired_ci(diffs)
        row[f"Δ_{m}"] = round(mean, 3)
        row[f"Δ_{m}_ci_lo"] = round(lo, 3)
        row[f"Δ_{m}_ci_hi"] = round(hi, 3)
        row[f"Δ_{m}_reliable"] = bool((lo > 0) or (hi < 0))
        row[f"Δ_{m}_n"] = n
    filter_rows.append(row)

filter_df = pd.DataFrame(filter_rows)
out_path = f"{SAVE_DIR}/filter_effect.csv"
filter_df.to_csv(out_path, index=False)
print(f"✓ Saved: {out_path}")
display(filter_df)


In [49]:
# --- 6.5  Ranking table — top 15 configurations by Sharpe ---
for w in WINDOWS_TO_RUN:
    print(f"\n=== {w.upper()} — Top 15 by Return ===")
    ranking_df = tbl.make_ranking_table(metrics_df, w, metric="Return%", n=15, save=False)
    display(ranking_df)


=== W1 — Top 15 by Return ===


,Dataset,Algo,Return%,CAGR,MaxDD%,Volatility%,Corr_DJIA,Corr_all,Seeds
Rank,,,,,,,,,
1,flt_mvx2,sac,36.41,44.40,-14.61,27.49,0.933,0.952,5.0
2,flt_mvx2,sac,36.41,44.40,-14.61,27.49,0.933,0.952,5.0
3,bollx_trg,ppo,36.01,43.89,-9.22,17.84,0.683,0.958,5.0
4,flt_mvx2_trg,sac,34.32,41.78,-16.72,28.64,0.939,0.943,5.0
5,flt_mvx2,a2c,34.15,41.57,-21.39,33.18,0.963,0.981,5.0
6,flt_mvx2,a2c,34.15,41.57,-21.39,33.18,0.963,0.981,5.0
7,flt_bollx,a2c,33.67,40.96,-20.04,30.59,0.947,0.973,5.0
8,all,ppo,33.21,40.39,-9.18,20.64,0.802,1.000,5.0
9,flt_mvx2_trg,a2c,32.33,39.29,-22.05,34.19,0.958,0.974,5.0



=== W4 — Top 15 by Return ===


,Dataset,Algo,Return%,CAGR,MaxDD%,Volatility%,Corr_DJIA,Corr_all,Seeds
Rank,,,,,,,,,
1,bollx_trg,sac,3.74,3.75,-14.88,18.21,0.944,NaN,5.0
2,flt_mvx2,ppo,-3.09,-3.10,-16.02,16.56,0.957,NaN,5.0
3,flt_mvx2,ppo,-3.09,-3.10,-16.02,16.56,0.957,NaN,5.0
4,flt_mvx2_trg,ppo,-3.18,-3.19,-15.48,17.75,0.952,NaN,5.0
5,bollx,ppo,-3.46,-3.48,-17.42,19.70,0.979,NaN,5.0
6,mvx2,ppo,-4.32,-4.34,-19.39,20.55,0.975,NaN,5.0
7,mvx2_trg,ppo,-4.63,-4.65,-19.03,21.16,0.975,NaN,5.0
8,bollx,sac,-5.04,-5.06,-16.90,18.97,0.959,NaN,5.0
9,bollx_trg,ppo,-5.26,-5.28,-18.28,20.85,0.981,NaN,5.0



=== W3 — Top 15 by Return ===


,Dataset,Algo,Return%,CAGR,MaxDD%,Volatility%,Corr_DJIA,Corr_all,Seeds
Rank,,,,,,,,,
1,flt_mvx2_trg,ppo,24.27,24.48,-11.98,14.92,0.939,0.951,5.0
2,all,ppo,21.19,21.47,-17.50,17.18,0.949,1.000,5.0
3,mvx2,ppo,21.01,21.20,-18.76,18.63,0.960,0.977,5.0
4,mv2,ppo,19.36,19.53,-16.94,16.56,0.959,0.986,5.0
5,flt_mvx2,sac,17.31,17.46,-18.72,17.70,0.977,0.977,5.0
6,flt_mvx2,sac,17.31,17.46,-18.72,17.70,0.977,0.977,5.0
7,flt_mvx2,ppo,16.90,17.05,-10.41,13.92,0.907,0.834,5.0
8,flt_mvx2,ppo,16.90,17.05,-10.41,13.92,0.907,0.834,5.0
9,flt_bollx,sac,16.78,16.93,-14.75,16.32,0.979,0.931,5.0


In [50]:
# --- 6.5  Ranking table — top 15 configurations by Sharpe ---
for w in WINDOWS_TO_RUN:
    print(f"\n=== {w.upper()} — Top 15 by Sharpe ===")
    ranking_df = tbl.make_ranking_table(metrics_df, w, metric="sharpe", n=15, save=False)
    display(ranking_df)


=== W1 — Top 15 by Sharpe ===


,Dataset,Algo,Sharpe,CAGR,Return%,MaxDD%,Volatility%,Corr_DJIA,Corr_all,Seeds
Rank,,,,,,,,,,
1,bollx_trg,ppo,2.139,43.89,36.01,-9.22,17.84,0.683,0.958,5.0
2,flt_mvx2,ppo,1.958,27.01,22.40,-6.95,12.68,0.674,0.899,5.0
3,flt_mvx2,ppo,1.958,27.01,22.40,-6.95,12.68,0.674,0.899,5.0
4,mvx2_trg,ppo,1.846,37.19,30.64,-9.93,18.10,0.692,0.955,5.0
5,mvx2,ppo,1.824,38.16,31.42,-9.39,18.78,0.727,0.971,5.0
6,mv2,ppo,1.797,37.68,31.04,-9.89,18.88,0.724,0.957,5.0
7,all,ppo,1.755,40.39,33.21,-9.18,20.64,0.802,1.000,5.0
8,flt_bollx,ppo,1.701,25.43,21.11,-8.20,13.96,0.604,0.862,5.0
9,bollx,ppo,1.690,35.06,28.92,-10.01,18.93,0.710,0.960,5.0



=== W4 — Top 15 by Sharpe ===


,Dataset,Algo,Sharpe,CAGR,Return%,MaxDD%,Volatility%,Corr_DJIA,Corr_all,Seeds
Rank,,,,,,,,,,
1,bollx_trg,sac,0.293,3.75,3.74,-14.88,18.21,0.944,NaN,5.0
2,bollx,ppo,-0.083,-3.48,-3.46,-17.42,19.70,0.979,NaN,5.0
3,flt_mvx2_trg,ppo,-0.095,-3.19,-3.18,-15.48,17.75,0.952,NaN,5.0
4,flt_mvx2,ppo,-0.109,-3.10,-3.09,-16.02,16.56,0.957,NaN,5.0
5,flt_mvx2,ppo,-0.109,-3.10,-3.09,-16.02,16.56,0.957,NaN,5.0
6,mvx2,ppo,-0.115,-4.34,-4.32,-19.39,20.55,0.975,NaN,5.0
7,mvx2_trg,ppo,-0.121,-4.65,-4.63,-19.03,21.16,0.975,NaN,5.0
8,bollx_trg,ppo,-0.158,-5.28,-5.26,-18.28,20.85,0.981,NaN,5.0
9,bollx,sac,-0.181,-5.06,-5.04,-16.90,18.97,0.959,NaN,5.0



=== W3 — Top 15 by Sharpe ===


,Dataset,Algo,Sharpe,CAGR,Return%,MaxDD%,Volatility%,Corr_DJIA,Corr_all,Seeds
Rank,,,,,,,,,,
1,flt_mvx2_trg,ppo,1.548,24.48,24.27,-11.98,14.92,0.939,0.951,5.0
2,all,ppo,1.222,21.47,21.19,-17.50,17.18,0.949,1.000,5.0
3,flt_mvx2,ppo,1.205,17.05,16.90,-10.41,13.92,0.907,0.834,5.0
4,flt_mvx2,ppo,1.205,17.05,16.90,-10.41,13.92,0.907,0.834,5.0
5,mv2,ppo,1.163,19.53,19.36,-16.94,16.56,0.959,0.986,5.0
6,mvx2,ppo,1.128,21.20,21.01,-18.76,18.63,0.960,0.977,5.0
7,flt_bollx,sac,1.043,16.93,16.78,-14.75,16.32,0.979,0.931,5.0
8,flt_mvx2,sac,1.000,17.46,17.31,-18.72,17.70,0.977,0.977,5.0
9,flt_mvx2,sac,1.000,17.46,17.31,-18.72,17.70,0.977,0.977,5.0


In [ ]:
# --- 6.6  Hypothesis evidence table ---
hyp_df = tbl.make_hypothesis_table(metrics_df, WINDOWS_TO_RUN, save=SAVE_TABLE, save_dir = SAVE_DIR )
display(hyp_df)

In [25]:
# --- 6.7a  State space reduction table (indicator count & state size vs `all` baseline) ---
for w in WINDOWS_TO_RUN:
    ss_df = tbl.make_state_space_table(
        datasets  = DATASETS,
        data      = data,
        window    = w,
        stock_dim = 30,
        save=SAVE_TABLE, 
        save_dir = SAVE_DIR +f"/{w}",
    )
    ss_df

Saved: tables_mv2boll/w1/state_space_reduction.csv
Saved: tables_mv2boll/w4/state_space_reduction.csv
Saved: tables_mv2boll/w3/state_space_reduction.csv


In [26]:
# --- 6.7c  Data reduction table (actual timesteps × indicator cols vs `all` baseline) ---
for w in WINDOWS_TO_RUN:
    data_reduc_df = tbl.make_data_reduction_table(
        datasets = DATASETS,
        data     = data,
        windows  = w,
        stock_dim = 30,
        save=SAVE_TABLE, save_dir = SAVE_DIR +f"/{w}",
    )
    data_reduc_df

Saved: tables_mv2boll/w1/data_reduction.csv
Saved: tables_mv2boll/w4/data_reduction.csv
Saved: tables_mv2boll/w3/data_reduction.csv


In [27]:
# --- 6.7b  Build price_store for trade statistics ---
# price_store[window][dataset] = FinRL-format test DataFrame (date, tic, close, ...)
price_store = {
    w: {
        ds: data[w][ds]["test"]["df"]
        for ds in DATASETS
        if ds in data.get(w, {})
    }
    for w in WINDOWS_TO_RUN
}
print("price_store built:",
      {w: len(v) for w, v in price_store.items()}, "datasets per window")

price_store built: {'w1': 9, 'w4': 9, 'w3': 9} datasets per window


In [28]:
# --- 6.8  Trade statistics table (counts + FIFO win rates, seed=42 representative) ---
trade_df = tbl.make_trade_stats_table(
    actions_store = actions_store,
    price_store   = price_store,
    windows       = WINDOWS_TO_RUN,
    datasets      = DATASETS,
    algos         = list(ALGOS),
    save=SAVE_TABLE, 
    save_dir = SAVE_DIR,
)
display(trade_df)

Saved: tables_mv2boll/trade_stats.csv


Total Trades  Buys  Sells  Long Days  Short Days  \
window dataset      algo                                                     
w1     all          a2c            199   176     23       2544           0   
                    ppo           3263  1956   1307       4171           0   
                    sac            890   548    342       2699           0   
       boll         a2c             98    98      0       3180           0   
                    ppo           1422  1051    371       3240           0   
...                                ...   ...    ...        ...         ...   
w3     mvx2_trg     ppo           1070   760    310       3462           0   
                    sac            210   134     76       2424           0   
       flt_mvx2_trg a2c             95    69     26       1236           0   
                    ppo            969   655    314       1686           0   
                    sac            190   127     63       1339           0   

                          Buy Win%  Sell Win%  
window dataset      algo                       
w1     all          a2c      93.64      52.17  
                    ppo      69.86      77.12  
                    sac      84.78      54.97  
       boll         a2c      93.88        NaN  
                    ppo      91.04      78.44  
...                            ...        ...  
w3     mvx2_trg     ppo      74.72      59.68  
                    sac      63.01      46.05  
       flt_mvx2_trg a2c      76.67      84.62  
                    ppo      89.51      57.01  
                    sac      70.41      55.56  

[81 rows x 7 columns]

In [ ]:
# --- 6.9  Trade distribution heatmaps (total / buys / sells by ticker) ---
for w in WINDOWS_TO_RUN:
    for metric in ["total", "buys", "sells"]:
        plot_trade_heatmap(
            actions_store = actions_store,
            window        = w,
            datasets      = DATASETS,
            algos         = list(ALGOS),
            metric        = metric,
            save=SAVE_TABLE, 
            save_dir = SAVE_DIR,
        )

In [ ]:
# --- 6.9b  Aggregate buy/sell distribution by ticker (summed over all datasets, per algo) ---
for w in WINDOWS_TO_RUN:
    plot_ticker_buysell_aggregated(
        actions_store = actions_store,
        window        = w,
        datasets      = DATASETS,
        algos         = list(ALGOS),
        save=SAVE_TABLE,
        save_dir = SAVE_DIR +f"/{w}",
    )

In [ ]:
# --- 6.10  Top-5 configurations equity curves (by Return%, ±1σ, with baselines) ---
importlib.reload(tbl)
for w in WINDOWS_TO_RUN:
    plot_top_configs_equity(
        agg              = agg,
        window           = w,
        datasets         = DATASETS,
        algos            = list(ALGOS),
        top_n            = 10,
        metric           = "Return%",
        metrics_df       = metrics_df.loc["w1"],
        baselines        = baselines,
        exclude_datasets = [
            
        ],
       save=SAVE_TABLE, save_dir = "imgs/dow/all"
    )

In [ ]:
# --- 6.11  Generate reports/results.md ---
importlib.reload(tbl)

_ranking_for_report = tbl.make_ranking_table(metrics_df, WINDOWS_TO_RUN[0], metric="sharpe", n=15)

tbl.generate_results_md(
    metrics_df        = metrics_df,
    hypothesis_df     = hyp_df,
    ranking_df        = _ranking_for_report,
    windows           = WINDOWS_TO_RUN,
    trade_stats_df    = trade_df,
    baselines         = baselines,
    state_space_df    = ss_df,          # from cell ss_reduction_tbl (6.7a)
    data_reduc_df     = data_reduc_df,  # from cell data_reduction_tbl (6.7c)
    output_path       = "../../reports/results.md",
    img_base          = "../FinRLCode/myresearch/imgs/dow",
)
print("results.md written.")

results.md written → /home/main/Dropbox/phd/ResearchFinRL/reports/results.md
results.md written.


## 7 — Interpretability: Where Does the Agent Trade?

Shows the A2C agent's buy/sell actions for AAPL during the W1 test period (2020-03 → 2021-01),
overlaid on the closing price and indicator triggers.

Green triangles = agent increases position (buy). Red triangles = agent decreases position (sell).
Dashed vertical lines = indicator trigger events (green = bullish, red = bearish).
Lower panel = shares held over time.

In [31]:
SAVE_DIR =  f"imgs/dow/presentation"

In [ ]:
# --- 7.1  A2C + MVX_TRG — AAPL (trigger-only state, MA overlay auto-detected) ---

_w  = WINDOWS_TO_RUN[0]   # w1
_ds = "mvx_trg"

if (_w in actions_store and _ds in actions_store[_w]
        and "a2c" in actions_store[_w][_ds]):
    plot_rl_trades_on_price(
        actions_df          = actions_store[_w][_ds]["a2c"],
        price_df            = data[_w][_ds]["test"]["df"],
        tic                 = "AAPL",
        auto_detect_ma      = True,
        auto_detect_signals = True,
        title               = "PPO MVX SUBSAMPLE— AAPL Trades (W1 test period)",
        save=SAVE_TABLE, save_dir = SAVE_DIR,
    )
else:
    print(f"[SKIP] No actions stored for {_w}/{_ds}/a2c")

In [ ]:
# --- 7.2  A2C + MVX2_TRG — AAPL (trigger-only state, MA overlay auto-detected) ---
_ds = "mvx2_trg"

if (_w in actions_store and _ds in actions_store[_w]
        and "a2c" in actions_store[_w][_ds]):
    plot_rl_trades_on_price(
        actions_df          = actions_store[_w][_ds]["a2c"],
        price_df            = data[_w][_ds]["test"]["df"],
        tic                 = "AAPL",
        auto_detect_ma      = True,
        auto_detect_signals = True,
        title               = "A2C + MVX2_TRG — AAPL Trades (W1 test period)",
        save=SAVE_TABLE, save_dir = SAVE_DIR,
    )
else:
    print(f"[SKIP] No actions stored for {_w}/{_ds}/a2c")

In [ ]:
# --- 7.3  PPO + flt_mvx2 across multiple tickers (with cross-ticker signal visibility) ---
%matplotlib inline
_w  = WINDOWS_TO_RUN[0]   # w1
_ds      = "flt_mvx2"
_algo    = "ppo"
_tickers = ["AAPL", "MSFT", "JPM", "GS"]

# Get ALL tickers from the data, not just the 4 selected
_all_tickers_in_data = sorted(data[_w][_ds]["test"]["df"]["tic"].unique())
print(f"All {len(_all_tickers_in_data)} tickers: {_all_tickers_in_data}")
for _tic in _tickers[:1]:  # Test AAPL first
    _others = {t: data[_w][_ds]["test"]["df"] for t in _all_tickers_in_data if t != _tic}
    
    # Extract actions for this ticker - CORRECT format
    actions_raw = actions_store[_w][_ds][_algo][[_tic]]  # Shape: (81, 1)
    
    # Reset index to make 'date' a column
    _tic_actions = actions_raw.reset_index()  # Now has ['date', ticker_name]
    print(f"After reset_index: {_tic_actions.shape}, columns: {list(_tic_actions.columns)}")
    
    # Rename ticker column to 'action'
    _tic_actions.columns = ['date', 'action']
    _tic_actions['date'] = pd.to_datetime(_tic_actions['date'])
    _tic_actions['tic'] = _tic
    
    print(f"Final _tic_actions:\n{_tic_actions.head()}")
    print(f"Columns: {list(_tic_actions.columns)}")
    
    # NOW call the plot function
    plot_rl_trades_on_price(
    actions_df=actions_store[_w][_ds][_algo],
    price_df=data[_w][_ds]["test"]["df"],
    tic="AAPL",
    show_ma=True,
    # NEW OPTIONS:
    only_ma_cross=False,  # Set to True to see only MA crosses
    show_cross_ticker_signals=False,  # Set to True to show other tickers' signals
    show_position_subplot=False, 
    save_dir = SAVE_DIR,
)




In [ ]:
# --- 7.3  PPO + flt_mvx2 across multiple tickers (with cross-ticker signal visibility) ---
%matplotlib inline
_w  = WINDOWS_TO_RUN[0]   # w1
_ds      = "flt_mvx2"
_algo    = "ppo"
_tickers = ["AAPL", "MSFT", "JPM", "GS"]

# Get ALL tickers from the data, not just the 4 selected
_all_tickers_in_data = sorted(data[_w][_ds]["test"]["df"]["tic"].unique())
print(f"All {len(_all_tickers_in_data)} tickers: {_all_tickers_in_data}")

for _tic in _tickers:
    # Show triggers from all OTHER tickers (all 29 others when plotting each one)
    _others = {t: data[_w][_ds]["test"]["df"] for t in _all_tickers_in_data if t != _tic}
    
    if (_w in actions_store and _ds in actions_store[_w]
            and _algo in actions_store[_w][_ds]):
        plot_rl_trades_on_price(
            actions_df             = actions_store[_w][_ds][_algo],
            price_df               = data[_w][_ds]["test"]["df"],
            tic                    = _tic,
            auto_detect_ma         = True,
            auto_detect_signals    = True,
            show_ma                = True,
            only_ma_cross=False,
            show_cross_ticker_signals=True,
            cross_ticker_price_df  = _others,
            algo                   = _algo,
            dataset                = _ds,
            title                  = f"PPO SUBSAMPLE MVX2 — {_tic} Trades (W1 test period)",
            save=SAVE_TABLE, save_dir = SAVE_DIR,
            show_position_subplot=False, 
        )
    else:
        print(f"[SKIP] No actions for {_w}/{_ds}/{_algo}")


In [ ]:
metrics_df.loc["w1"]

In [ ]:
# --- 7.4  Top configurations equity comparison (top 10 by Sharpe, ±1σ) ---
for w in WINDOWS_TO_RUN:
    plot_top_configs_equity(
        agg              = agg,
        window           = w,
        datasets         = DATASETS,
        algos            = list(ALGOS),
        top_n            = 10,
        metrics_df       = metrics_df.loc[w],
        baselines        = baselines,
        exclude_datasets = [ ],
        save             = True,
        save_dir         = "imgs/dow/all",
    )